In [2]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# In[2]: Load sample data and create vector store
financial_news = [
    "Apple beats earnings expectations with strong services growth",
    "Google announces breakthrough in quantum computing research",
    "Amazon faces regulatory scrutiny in EU markets",
    "Netflix subscriber growth slows in saturated markets",
    "Meta invests heavily in metaverse development despite losses"
]

embed = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = FAISS.from_texts(financial_news, embed)

# In[3]: Set up Ollama LLM
llm = OllamaLLM(model="llama3")

# In[4]: Create retrieval chain
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(retriever, question_answer_chain)

# In[5]: Test with a question
question = "Which company announced a breakthrough in quantum computing?"
response = qa_chain.invoke({"input": question})
print(f"Q: {question}")
print(f"A: {response['answer']}")

Q: Which company announced a breakthrough in quantum computing?
A: According to the given context, Google announced a breakthrough in quantum computing research.
